# TP1 (STARTER): Structured Outputs, Tool Use & Building a ReAct Agent from Scratch

**Course:** `4S4TWS_21` — Agentic AI (École Centrale de Lyon & emlyon business school)  
**Instructor:** Chouaieb Nemri  
**Session:** Day 1 Morning — Wednesday, October 14, 2026 (`10:00 – 12:00`)  
**SDK & Models:** `google-genai` (`gemma-4-26b-a4b-it` & `gemma-4-31b-it`)

---

## Learning Objectives
By the end of this 2-hour hands-on lab, you will be able to:
1. Enforce strict **Pydantic JSON schemas** on Gemma 4 / Gemini outputs so downstream code never crashes on malformed markdown.
2. Design **typed Python tools** with Google-style docstrings that serve as the LLM's API contract.
3. Understand the difference between **Automatic Function Calling (AFC)** and **Manual Function Calling**.
4. Build a transparent **ReAct (`Thought -> Action -> Observation`) loop** from scratch with iteration budgets, Free-Tier thought-token trimming, and automatic error recovery.

In [ ]:
# Step 0: Install & Import Dependencies
# Uncomment the line below if running in Google Colab:
# !pip install -q -U google-genai pydantic

import os
import json
import time
from pathlib import Path
from typing import Literal
from pydantic import BaseModel, Field
try:
    from google import genai
    from google.genai import types
except ImportError:
    genai = None
    types = None

# Load the course dataset (works both locally and in Colab)
DATA_PATHS = [
    Path("../data/supplier_esg_audit_db.json"),
    Path("03_Labs_TP/data/supplier_esg_audit_db.json"),
    Path("/Users/nemri/Downloads/ecl-agentic-ai-course/03_Labs_TP/data/supplier_esg_audit_db.json"),
]
for p in DATA_PATHS:
    if p.exists():
        with open(p, "r", encoding="utf-8") as f:
            SUPPLIER_DB = json.load(f)
        break

print(f"Loaded {len(SUPPLIER_DB['suppliers'])} suppliers from dataset.")
MODEL_ID = "gemma-4-26b-a4b-it"  # Or "gemma-4-31b-it"

---
## Exercise 1: Structured Outputs with Pydantic Schemas

Unstructured text is great for humans, but terrible for software pipelines. Before an agent can reliably route tasks or trigger business logic, we need **deterministic JSON outputs** validated by Pydantic.

### Task 1.1: Define the `SupplierQuickAssessment` Schema
Create a Pydantic model `SupplierQuickAssessment` with:
- `supplier_id`: `str`
- `company_name`: `str`
- `total_scope1_2_tco2e`: `float` (sum of Scope 1 and Scope 2 emissions)
- `risk_level`: `Literal["LOW_RISK", "MEDIUM_RISK", "HIGH_RISK"]`
- `key_findings`: `list[str]` (2 to 4 concise engineering/ESG findings)


In [ ]:
# TODO: Complete the SupplierQuickAssessment Pydantic model
class SupplierQuickAssessment(BaseModel):
    supplier_id: str = Field(description="Unique supplier identifier, e.g. SUP-001")
    # TODO: Add company_name (str), total_scope1_2_tco2e (float),
    # risk_level (Literal["LOW_RISK", "MEDIUM_RISK", "HIGH_RISK"]), and key_findings (list[str])
    pass

# TODO: Call client.models.generate_content with response_mime_type="application/json"
# and response_schema=SupplierQuickAssessment on SUPPLIER_DB["suppliers"][1]


---
## Exercise 2: Designing Typed Python Tools (The LLM's API Contract)

Remember the golden rule of Function Calling:
> **The LLM never executes your Python code directly.** It reads your function's **name**, **type hints**, and **docstring**, and emits a structured JSON `function_call` requesting that your application run the function with specific arguments.

### Task 2.1: Implement Three Deterministic Audit Tools
1. `lookup_supplier_emissions(supplier_id: str) -> dict`: Looks up a supplier by ID in `SUPPLIER_DB["suppliers"]`. Returns `{"status": "success", "supplier": ...}` or `{"status": "error", "error_message": ...}`.
2. `calculate_cbam_tax_liability(scope1_tco2e: float, scope2_tco2e: float, eu_cbam_covered: bool, allocation_factor: float = 0.15) -> dict`: Computes the EU Carbon Border Adjustment Mechanism (CBAM) liability at **€85.0 / tCO2e** (`(scope1 + scope2) * 85.0 * allocation_factor` if `eu_cbam_covered` is `True`, else `0.0`).
3. `check_csrd_compliance_status(annual_spend_eur: float, scope1_tco2e: float, scope2_tco2e: float, sbti_verified: bool, labor_safety_incidents_12m: int) -> dict`: Applies our Responsible Procurement Policy v4.2:
   - If `annual_spend_eur < 2_000_000`: exempt SME (`LOW_RISK`, `mandatory_cap=False`).
   - Else if `(scope1 + scope2 > 25_000 and not sbti_verified)` or `labor_safety_incidents_12m >= 3`: `HIGH_RISK` (`mandatory_cap=True`).
   - Else if `scope1 + scope2 > 20_000`: `MEDIUM_RISK` (`mandatory_cap=False`).
   - Else: `LOW_RISK` (`mandatory_cap=False`).


In [ ]:
# TODO: Implement lookup_supplier_emissions, calculate_cbam_tax_liability, and check_csrd_compliance_status
# Make sure every parameter has a type hint and a clear Google-style docstring!


---
## Exercise 3: Building the Transparent ReAct Loop from Scratch

While `google-genai` supports **Automatic Function Calling (AFC)** out of the box, production engineers must know how to build the **manual ReAct loop** (`AutomaticFunctionCallingConfig(disable=True)`) so you can:
- Log every tool call and latency metric for auditability
- Enforce a strict `max_iterations` budget so an agent never loops forever
- Catch tool exceptions and feed structured error messages back to the model so it can self-correct

### Task 3.1: Implement `run_manual_react_agent(prompt: str, max_iterations: int = 5)`


In [ ]:
def strip_thought_parts(content):
    """Strip internal `thought=True` blocks before appending to `contents` (~65% Free-Tier input token savings)."""
    if content is None or not getattr(content, "parts", None):
        return content
    clean_parts = [p for p in content.parts if not getattr(p, "thought", False)]
    return types.Content(role=content.role, parts=clean_parts)

# TODO: Implement run_manual_react_agent(prompt: str, max_iterations: int = 5)
# 1. Disable Automatic Function Calling so you control the loop
# 2. Append strip_thought_parts(response.candidates[0].content) on each iteration
# 3. Check response.function_calls on each iteration
# 4. Execute requested tools from TOOL_REGISTRY inside a try/except block
# 5. Append types.Part.from_function_response(...) back into contents

---
## Stretch Exercise (Bonus for Fast Finishers): Carbon Price Stress-Test (`€85` vs. `€110/ton`)

In real procurement negotiations, finance teams want to know what happens if EU ETS carbon prices spike from **€85/tCO2e** to **€110/tCO2e**.
Write a helper function `compare_carbon_price_scenarios(supplier_id: str, rates: list[float] = [85.0, 110.0]) -> dict` that looks up a supplier's emissions with `lookup_supplier_emissions` and calculates how much extra tax the company would pay at €110/ton vs. €85/ton.


In [ ]:
def compare_carbon_price_scenarios(supplier_id: str, rates: list[float] = [85.0, 110.0]) -> dict:
    # TODO (Stretch Task): Compare CBAM liability across multiple EUR/tCO2e carbon price scenarios
    pass
